Autor: **[Escribe tu nombre]**

# **Unsupervised Learning**
## **Búsqueda del valor óptimo de $k$ con K-Means**
### Dataset: Personas Desaparecidas

En este notebook trabajaremos con un **nuevo dataset almacenado en Excel** y aplicaremos **K-Means Clustering** para encontrar agrupaciones naturales en los datos.

El objetivo principal es:

> **Search the optimal value of $k$ for K-Means clustering on a new dataset.**

Para seleccionar un valor apropiado de $k$ utilizaremos principalmente:

- **Elbow Method (Método del Codo)**.
- **Silhouette Score**.
- Una visualización final de los clusters mediante **PCA**.


## **1. Introducción: ¿Qué es Unsupervised Learning?**

El **aprendizaje no supervisado** (*Unsupervised Learning*) es una rama de Machine Learning en la cual el algoritmo trabaja con datos que **no tienen una etiqueta objetivo conocida**.

A diferencia del aprendizaje supervisado, no existe una columna que le diga al modelo cuál es la respuesta correcta. El algoritmo debe descubrir patrones, estructuras o agrupaciones por sí mismo.

### **Clustering**

Una de las tareas más comunes del aprendizaje no supervisado es el **clustering**, que consiste en agrupar observaciones similares.

### **K-Means**

K-Means divide los datos en $k$ grupos. De forma simplificada:

1. Se seleccionan $k$ centroides iniciales.
2. Cada observación se asigna al centroide más cercano.
3. Se recalcula el centro de cada grupo.
4. El proceso se repite hasta que las asignaciones se estabilizan.

El valor de $k$ debe decidirse antes de entrenar el modelo. Por eso analizaremos varios valores posibles.


## **2. Contenido**

1. Importación de librerías.
2. Carga del archivo Excel.
3. Exploración inicial.
4. Limpieza del dataset.
5. Identificación de variables numéricas y categóricas.
6. Preprocesamiento.
7. Aplicación inicial de K-Means.
8. Búsqueda del valor óptimo de $k$.
9. Elbow Method.
10. Silhouette Score.
11. Modelo K-Means final.
12. Visualización de clusters con PCA.
13. Perfil de los clusters.
14. Exportación de resultados.
15. Preguntas de análisis.


### **2.1 Instalación de paquetes**

Ejecuta esta celda solamente si todavía no tienes instaladas estas librerías.

In [ ]:
%pip install pandas numpy matplotlib scikit-learn openpyxl

### **2.2 Importamos los paquetes necesarios**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print("Librerías importadas correctamente.")

## **3. Cargamos la base de datos**

El archivo utilizado en este ejercicio está en formato **Excel (`.xlsx`)**.

La ruta está escrita como *raw string* usando `r"..."` para evitar problemas con las barras `\` de Windows.

También se muestra, como comentario, cómo se cargaría un archivo `.txt`.


In [ ]:
# ============================================================
# ARCHIVO EXCEL
# ============================================================

archivo = r"C:\Documentos\2SEM2026\intelligence Artificial\lab assigment 1\mdi_personasdesaparecidas_pm_2026_enero_julio.xlsx"

# ============================================================
# SI EL ARCHIVO FUERA TXT
# ============================================================

# TXT separado por tabulaciones:
# data = pd.read_csv(r"C:\Documentos\dataset.txt", sep="\t")

# TXT separado por comas:
# data = pd.read_csv(r"C:\Documentos\dataset.txt", sep=",")


### **3.1 Revisamos las hojas disponibles**

Este paso es importante porque algunos archivos Excel contienen una primera hoja informativa y los datos reales se encuentran en otra pestaña.

In [ ]:
excel = pd.ExcelFile(archivo)

print("Hojas disponibles:")
for i, hoja in enumerate(excel.sheet_names):
    print(i, "->", hoja)


### **3.2 Seleccionamos automáticamente la hoja de datos**

El código busca una hoja cuyo nombre contenga `pdesaparecidas`. Si no la encuentra, utilizará la segunda hoja del libro cuando exista.

In [ ]:
sheet_name = None

for hoja in excel.sheet_names:
    if "pdesaparecidas" in hoja.lower():
        sheet_name = hoja
        break

if sheet_name is None:
    if len(excel.sheet_names) > 1:
        sheet_name = excel.sheet_names[1]
    else:
        sheet_name = excel.sheet_names[0]

print("Hoja seleccionada:", sheet_name)

data = pd.read_excel(
    archivo,
    sheet_name=sheet_name
)

print("Archivo leído correctamente.")


## **4. Exploración inicial del dataset**

Antes de aplicar Machine Learning debemos conocer la estructura de los datos.

In [ ]:
print("Dimensiones originales:", data.shape)
print("\nPrimeras 10 filas:")
display(data.head(10))


In [ ]:
print("Información del dataset:")
data.info()


In [ ]:
print("Columnas originales:")
for columna in data.columns:
    print("-", columna)


## **5. Limpieza del dataset**

Eliminaremos:

- Filas completamente vacías.
- Columnas completamente vacías.
- Columnas cuyo nombre empiece por `Unnamed`.
- Espacios innecesarios en los nombres de las columnas.


In [ ]:
# Creamos una copia para conservar los datos originales
data_clean = data.copy()

# Eliminamos filas y columnas completamente vacías
data_clean = data_clean.dropna(axis=0, how="all")
data_clean = data_clean.dropna(axis=1, how="all")

# Limpiamos los nombres de las columnas
data_clean.columns = [
    str(columna).strip()
    for columna in data_clean.columns
]

# Eliminamos columnas "Unnamed"
columnas_unnamed = [
    columna
    for columna in data_clean.columns
    if columna.lower().startswith("unnamed")
]

data_clean = data_clean.drop(
    columns=columnas_unnamed,
    errors="ignore"
)

data_clean = data_clean.reset_index(drop=True)

print("Dimensiones después de la limpieza:", data_clean.shape)
display(data_clean.head(10))


### **5.1 Intentamos reconocer números almacenados como texto**

Excel puede guardar algunas columnas numéricas con tipo `object`. El siguiente bloque intenta convertirlas cuando al menos el 80% de sus valores válidos parecen ser números.

In [ ]:
for columna in data_clean.columns:

    if data_clean[columna].dtype == "object":

        convertido = pd.to_numeric(
            data_clean[columna],
            errors="coerce"
        )

        originales_validos = data_clean[columna].notna().sum()
        convertidos_validos = convertido.notna().sum()

        if originales_validos > 0:

            proporcion = convertidos_validos / originales_validos

            if proporcion >= 0.80:
                data_clean[columna] = convertido

print("Conversión terminada.")


## **6. Identificamos los tipos de variables**

K-Means utiliza distancias, por lo que las variables deben convertirse a una representación numérica.

- Las variables **numéricas** serán imputadas y estandarizadas.
- Las variables **categóricas** serán imputadas y codificadas con `OneHotEncoder`.
- Las columnas de fecha se mostrarán, pero no se utilizarán directamente en este modelo.


In [ ]:
columnas_numericas = data_clean.select_dtypes(
    include=np.number
).columns.tolist()

columnas_categoricas = data_clean.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

columnas_fecha = data_clean.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()

print("Columnas numéricas:", columnas_numericas)
print("\nColumnas categóricas:", columnas_categoricas)
print("\nColumnas de fecha:", columnas_fecha)


## **7. Eliminamos columnas poco útiles para clustering**

Una columna como un ID, código único, nombre completo o número de expediente puede tener casi un valor diferente para cada fila. Ese tipo de variable puede distorsionar K-Means.

El siguiente bloque detecta automáticamente columnas categóricas de **alta cardinalidad**.

> Después de ejecutar la celda, revisa la lista de columnas eliminadas para asegurarte de que tenga sentido para tu dataset.


In [ ]:
columnas_alta_cardinalidad = []

for columna in columnas_categoricas:

    valores_unicos = data_clean[columna].nunique(dropna=True)
    valores_validos = data_clean[columna].notna().sum()

    if valores_validos > 0:

        proporcion_unicos = valores_unicos / valores_validos

        if valores_unicos > 30 and proporcion_unicos > 0.70:
            columnas_alta_cardinalidad.append(columna)

print("Columnas categóricas de alta cardinalidad detectadas:")
print(columnas_alta_cardinalidad)


### **7.1 Columnas que queremos excluir manualmente**

Si sabes que alguna columna corresponde a un identificador, código, nombre, dirección u otra variable que no quieres usar, agrégala en la lista `columnas_excluir_manual`.

Ejemplo:

```python
columnas_excluir_manual = ["ID", "NOMBRE"]
```


In [ ]:
# Puedes editar esta lista si es necesario
columnas_excluir_manual = []

columnas_excluir = list(
    set(
        columnas_alta_cardinalidad
        + columnas_fecha
        + columnas_excluir_manual
    )
)

data_modelo = data_clean.drop(
    columns=columnas_excluir,
    errors="ignore"
)

print("Columnas excluidas:")
print(columnas_excluir)

print("\nDimensiones del dataset utilizado por K-Means:")
print(data_modelo.shape)


## **8. Preprocesamiento**

### Variables numéricas

1. Los valores faltantes se reemplazan por la **mediana**.
2. Se aplica `StandardScaler` para que todas las variables tengan una escala comparable.

La estandarización es importante porque K-Means trabaja con distancias.

### Variables categóricas

1. Los valores faltantes se reemplazan por la categoría más frecuente.
2. `OneHotEncoder` transforma cada categoría en variables binarias.


In [ ]:
columnas_numericas_modelo = data_modelo.select_dtypes(
    include=np.number
).columns.tolist()

columnas_categoricas_modelo = data_modelo.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Variables numéricas usadas por el modelo:")
print(columnas_numericas_modelo)

print("\nVariables categóricas usadas por el modelo:")
print(columnas_categoricas_modelo)


In [ ]:
pipeline_numerico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

pipeline_categorico = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


In [ ]:
transformadores = []

if len(columnas_numericas_modelo) > 0:
    transformadores.append(
        (
            "numerico",
            pipeline_numerico,
            columnas_numericas_modelo
        )
    )

if len(columnas_categoricas_modelo) > 0:
    transformadores.append(
        (
            "categorico",
            pipeline_categorico,
            columnas_categoricas_modelo
        )
    )

if len(transformadores) == 0:
    raise ValueError(
        "No se encontraron variables útiles para aplicar K-Means."
    )

preprocesador = ColumnTransformer(
    transformers=transformadores
)

X = preprocesador.fit_transform(data_modelo)

print("Dimensiones antes del preprocesamiento:", data_modelo.shape)
print("Dimensiones después del preprocesamiento:", X.shape)


## **9. Primera prueba con K-Means**

Antes de buscar el mejor $k$, podemos observar qué ocurre con un valor inicial, por ejemplo:

$$k=2$$

Este valor es solamente una demostración y **no significa que sea el valor óptimo**.


In [ ]:
modelo_inicial = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

clusters_iniciales = modelo_inicial.fit_predict(X)

print("Inercia para k = 2:", modelo_inicial.inertia_)
print("Primeras etiquetas:", clusters_iniciales[:20])


## **10. Trabajo principal: buscar el valor óptimo de $k$**

Probaremos varios valores de:

$$k = 2,3,4,\ldots,10$$

Para cada valor calcularemos:

### **Inertia**

La inercia mide la suma de las distancias cuadráticas de las observaciones a sus centroides.

Una inercia menor indica grupos más compactos, pero la inercia siempre tiende a disminuir cuando aumentamos $k$.

### **Silhouette Score**

El Silhouette Score compara:

- Qué tan cerca está una observación de su propio cluster.
- Qué tan separada está de otros clusters.

Su rango aproximado es:

$$-1 \leq s \leq 1$$

Un valor más alto suele indicar una mejor estructura de clustering.


In [ ]:
numero_muestras = X.shape[0]

if numero_muestras < 3:
    raise ValueError(
        "El dataset tiene menos de 3 observaciones y no permite realizar esta búsqueda."
    )

max_k = min(10, numero_muestras - 1)
valores_k = list(range(2, max_k + 1))

inercias = []
silhouettes = []

print("Resultados:")
print("-" * 65)

for k in valores_k:

    modelo = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    etiquetas = modelo.fit_predict(X)

    inercia = modelo.inertia_
    inercias.append(inercia)

    numero_clusters_reales = len(np.unique(etiquetas))

    if 1 < numero_clusters_reales < numero_muestras:
        silhouette = silhouette_score(X, etiquetas)
    else:
        silhouette = np.nan

    silhouettes.append(silhouette)

    print(
        f"k = {k:2d} | "
        f"Inertia = {inercia:12.2f} | "
        f"Silhouette = {silhouette:.4f}"
    )


## **11. Elbow Method (Método del Codo)**

Buscamos un punto donde la disminución de la inercia deje de ser tan pronunciada. Ese cambio de pendiente forma visualmente un **codo**.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    valores_k,
    inercias,
    marker="o"
)

plt.xlabel("Número de clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(valores_k)
plt.grid()

plt.show()


## **12. Silhouette Method**

En esta gráfica buscamos el valor de $k$ con el **Silhouette Score más alto**.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    valores_k,
    silhouettes,
    marker="o"
)

plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Method")
plt.xticks(valores_k)
plt.grid()

plt.show()


### **12.1 Selección automática del mejor $k$**

En este notebook se selecciona automáticamente el valor con el mayor Silhouette Score.

El resultado debe compararse también con la gráfica del método del codo y con la interpretación del problema.


In [ ]:
silhouettes_array = np.array(
    silhouettes,
    dtype=float
)

if np.all(np.isnan(silhouettes_array)):
    raise ValueError(
        "No fue posible calcular un Silhouette Score válido."
    )

posicion_mejor = np.nanargmax(silhouettes_array)

mejor_k = valores_k[posicion_mejor]
mejor_silhouette = silhouettes[posicion_mejor]

print("=" * 50)
print("RESULTADO")
print("=" * 50)
print("Mejor valor de k según Silhouette Score:", mejor_k)
print("Mejor Silhouette Score:", round(mejor_silhouette, 4))


## **13. Entrenamos el modelo final**

Ahora utilizamos el valor de $k$ seleccionado para entrenar el modelo K-Means definitivo.


In [ ]:
modelo_kmeans = KMeans(
    n_clusters=mejor_k,
    random_state=42,
    n_init=10
)

clusters = modelo_kmeans.fit_predict(X)

resultado = data_clean.copy()
resultado["Cluster"] = clusters

display(resultado.head(20))


### **13.1 Número de observaciones en cada cluster**

In [ ]:
conteo_clusters = (
    resultado["Cluster"]
    .value_counts()
    .sort_index()
)

print("Cantidad de observaciones por cluster:")
display(conteo_clusters.to_frame("Cantidad"))


## **14. Visualización de los clusters usando PCA**

Después del preprocesamiento podemos tener muchas variables.

**PCA (Principal Component Analysis)** permite reducir los datos a dos dimensiones para crear una representación visual.

> Importante: PCA se utiliza aquí para visualizar. K-Means fue entrenado con todas las variables preprocesadas.


In [ ]:
if X.shape[1] >= 2:

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    plt.figure(figsize=(8, 6))

    scatter = plt.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=clusters
    )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(f"K-Means Clusters (k = {mejor_k})")
    plt.colorbar(
        scatter,
        label="Cluster"
    )
    plt.grid()

    plt.show()

    print(
        "Varianza explicada por los dos componentes:",
        round(pca.explained_variance_ratio_.sum(), 4)
    )

else:

    print(
        "No hay suficientes variables para realizar una visualización PCA en 2D."
    )


## **15. Perfil numérico de los clusters**

Para interpretar los grupos podemos comparar las medias de las variables numéricas originales.

Esto ayuda a responder preguntas como:

- ¿Qué características distinguen un cluster de otro?
- ¿Hay grupos con valores particularmente altos o bajos?
- ¿Qué patrón podría representar cada cluster?


In [ ]:
columnas_numericas_resultado = [
    columna
    for columna in columnas_numericas_modelo
    if columna in resultado.columns
]

if len(columnas_numericas_resultado) > 0:

    perfil_clusters = resultado.groupby(
        "Cluster"
    )[columnas_numericas_resultado].mean()

    display(perfil_clusters)

else:

    print(
        "No hay columnas numéricas originales disponibles para crear este perfil."
    )


## **16. Guardamos los resultados**

El nuevo Excel incluirá una columna llamada `Cluster` con el grupo asignado a cada observación.


In [ ]:
archivo_salida = r"C:\Documentos\2SEM2026\intelligence Artificial\lab assigment 1\personas_desaparecidas_clusters.xlsx"

resultado.to_excel(
    archivo_salida,
    index=False
)

print("Archivo guardado correctamente en:")
print(archivo_salida)

# Si quisieras guardar los resultados como TXT:
#
# resultado.to_csv(
#     r"C:\Documentos\2SEM2026\intelligence Artificial\lab assigment 1\personas_desaparecidas_clusters.txt",
#     sep="\t",
#     index=False
# )


# **Preguntas de análisis**

### **1. ¿Cuál es el valor óptimo de $k$?**

Escribe tu respuesta después de ejecutar las gráficas y comparar el método del codo con el Silhouette Score.

**Respuesta:**  
...

---

### **2. ¿Por qué consideras que ese valor de $k$ es adecuado?**

Considera:

- La forma de la gráfica del codo.
- El valor del Silhouette Score.
- La separación visual entre grupos.
- Si los clusters tienen una interpretación razonable.

**Respuesta:**  
...

---

### **3. ¿Qué características distinguen a cada cluster?**

Utiliza la tabla de perfiles y la distribución de observaciones por cluster.

**Respuesta:**  
...

---

### **4. ¿Por qué es importante estandarizar los datos antes de aplicar K-Means?**

**Respuesta:**  
...

---

### **5. Conclusión**

Resume qué encontró el algoritmo y qué aprendiste sobre el uso de Unsupervised Learning y K-Means.

**Respuesta:**  
...


## **Resumen del procedimiento**

El flujo utilizado fue:

$$
\text{Excel}
\rightarrow
\text{Limpieza}
\rightarrow
\text{Preprocesamiento}
\rightarrow
\text{K-Means para diferentes } k
\rightarrow
\text{Elbow + Silhouette}
\rightarrow
\text{Mejor } k
\rightarrow
\text{Modelo final}
\rightarrow
\text{Interpretación}
$$

El aprendizaje no supervisado no recibe una respuesta correcta previamente etiquetada. En su lugar, K-Means intenta descubrir una estructura interna en los datos agrupando observaciones semejantes.
